# 🧩 Classic Memory: Buffer, Window, Summary and Vector Memory in Chains

## Learning Objectives
In this notebook, you will learn:
1. **The classic memory API** - what `ConversationBufferMemory` stores and what it hands to a prompt
2. **The load → invoke → save loop** - plug classic memory into an LCEL chain with `RunnablePassthrough.assign`
3. **Bounding strategies** - window, summary and summary-buffer memory, and what each lets the model see
4. **Vector-store memory** - retrieve the most *relevant* past turns instead of the most recent ones
5. **Legacy conversation chains** - how `ConversationChain` and `LLMChain` saved memory automatically, and what replaced them

## Where this fits
`01_why_llms_forget.ipynb` showed that memory is the list of messages you resend. These classes are LangChain's
original (0.x-era) strategies for deciding what goes into that list, and you will meet them in a lot of existing
code. Each class appears once: what it stores, then what the model sees when it sits in a chain. The manual
bookkeeping you will write is what `03_runnable_with_message_history.ipynb` automates.

## Prerequisites
- `01_why_llms_forget.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (Part 1 makes no API calls; Part 4 also uses OpenAI embeddings)

> **⚠️ Legacy API**: The classic memory classes (deprecated since 0.3.1), `ConversationChain` (0.2.7) and `LLMChain` (0.1.17) live in `langchain_classic` and are removed in `langchain-classic` 2.0. LangChain now recommends LangGraph persistence instead — a checkpointer
> and a `thread_id` — taught in [`01_memory_and_conversational_agent.ipynb`](../notebooks/06a_memory_and_conversational_agent.ipynb)
> (module 02). This module keeps the older API because existing LCEL code uses it everywhere; the setup cell
> lets its deprecation warnings through so you see them where the deprecated classes are used.


## 📋 Memory Architecture Overview

### Core Concepts
- **Memory buffer**: storage mechanism for conversation messages
- **Message history**: sequence of user and AI interactions
- **Memory variables**: processed memory content handed to prompt templates (`load_memory_variables`)
- **Context window**: the part of memory that is actually sent to the model

### Memory System Layers
- **Storage layer**: where messages are physically stored (`memory.chat_memory`)
- **Strategy layer**: how messages are selected (all, last *k*, summarised)
- **Processing layer**: how memory is formatted for the prompt (a string, or a list of messages with `return_messages=True`)

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library warnings — except LangChain's deprecation warnings, which it lets
through because this notebook uses deprecated APIs on purpose.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv
from langchain_core._api import LangChainDeprecationWarning

warnings.filterwarnings("ignore")
# This notebook uses deprecated LangChain APIs on purpose: let their deprecation warnings through
warnings.filterwarnings("default", category=LangChainDeprecationWarning)

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: answers every chain and writes the summaries
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧠 Part 1: The Classic Memory API — ConversationBufferMemory

`ConversationBufferMemory` retains every message in chronological order. It prioritises complete context
over efficiency.

### Key Concepts:
- **Complete retention**: every message is preserved, in order
- **No filtering**: all messages are returned on every read
- **Unbounded growth**: memory size grows with the conversation

**Use cases**: short to medium conversations, debugging and conversation analysis, prototypes.

In [ ]:
# ============================================================================
# PART 1: BUFFER MEMORY KEEPS EVERYTHING
# ============================================================================
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory()

# Add messages directly to the underlying chat history
memory.chat_memory.add_user_message("Hi, I'm Sourav from Databricks")
memory.chat_memory.add_ai_message("Hello Sourav! Nice to meet you.")
memory.chat_memory.add_user_message("I work with Apache Spark")
memory.chat_memory.add_ai_message("That's great! Spark is powerful for big data.")

print("=== All messages in the buffer ===")
for i, message in enumerate(memory.chat_memory.messages, 1):
    print(f"{i}. {message.type}: {message.content}")

# What a prompt template would receive
print(f"\n=== load_memory_variables() -> 'history' (a string by default) ===\n{memory.load_memory_variables({})['history']}")

# return_messages=True returns message objects instead — what chat prompts with MessagesPlaceholder need
as_messages = ConversationBufferMemory(return_messages=True, chat_memory=memory.chat_memory)
print(f"\n=== With return_messages=True ===\n{as_messages.load_memory_variables({})['history']}")

`return_messages` only changes the **format**: a single `"Human: …\nAI: …"` string for completion-style
prompts, or a list of `HumanMessage`/`AIMessage` objects for chat prompts.

| Advantages | Limitations |
|---|---|
| ✅ No information loss | ❌ Unbounded memory growth |
| ✅ Simple and predictable | ❌ Can exceed the model's context window |
| ✅ Full history available for analysis | ❌ Every call pays for the whole history in tokens |

---
## 🔁 Part 2: The Load → Invoke → Save Loop

An LCEL chain has no memory slot. To use a classic memory object you:
1. **Load** the history with `memory.load_memory_variables(...)` and add it to the chain input
   (`RunnablePassthrough.assign` adds a key while passing the rest through unchanged)
2. **Invoke** the prompt and model
3. **Save** the exchange yourself with `memory.save_context(...)`

Forgetting step 3 is the classic bug: the chain runs, but the next turn has no history.

In [ ]:
# ============================================================================
# PART 2A: WHAT RunnablePassthrough.assign ADDS TO THE INPUT
# ============================================================================
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

buffer_memory = ConversationBufferMemory(return_messages=True)
buffer_memory.save_context({"query": "Hi, I'm Sourav"}, {"output": "Hello Sourav!"})

def load_history(_inputs):
    return buffer_memory.load_memory_variables({})["history"]

# The original keys pass through; 'history' is added next to them
print(RunnablePassthrough.assign(history=RunnableLambda(load_history)).invoke({"query": "What's my name?"}))

In [ ]:
# ============================================================================
# HELPER: ONE CONVERSATION TURN WITH CLASSIC MEMORY
# ============================================================================
def ask(chain, memory, question: str) -> str:
    """Invoke the chain, then save the exchange — classic memory never saves itself inside LCEL."""
    query = {"query": question}
    response = chain.invoke(query)
    memory.save_context(query, {"output": response.content})
    print(f"User: {question}\nAI:   {response.content}\n")
    return response.content


print("✅ ask(chain, memory, question) defined")

In [ ]:
# ============================================================================
# PART 2B: A CONVERSATION CHAIN WITH BUFFER MEMORY
# ============================================================================
SYS_PROMPT = "Act as a helpful assistant and give brief answers."

prompt = ChatPromptTemplate.from_messages([
    ("system", SYS_PROMPT),
    MessagesPlaceholder(variable_name="history"),  # classic memory's messages go here
    ("human", "{query}"),
])

buffer_memory = ConversationBufferMemory(return_messages=True)

buffer_chain = (
    RunnablePassthrough.assign(history=RunnableLambda(lambda _: buffer_memory.load_memory_variables({})["history"]))
    | prompt
    | llm
)

ask(buffer_chain, buffer_memory, "What are the first four colors of a rainbow?")
ask(buffer_chain, buffer_memory, "And the other 3?")  # only answerable with the previous turn
print(f"Messages in memory: {len(buffer_memory.chat_memory.messages)}")

"And the other 3?" only makes sense because the previous exchange was loaded into the prompt. The memory now
holds 4 messages — two per `ask` call, both written by our explicit `save_context`.

---
## 🪟 Part 3: Bounding the History

A buffer grows forever. The three classes below bound what reaches the prompt in different ways: drop old
exchanges (window), compress them (summary), or keep recent turns verbatim and compress only the overflow
(summary buffer). The chain from Part 2 stays the same; only the memory object changes.

### 3.1 🪟 Window memory
`ConversationBufferWindowMemory` keeps a sliding window of the most recent **k exchanges** — one exchange is a
human message plus the AI reply, so the window holds `2k` messages. The underlying `chat_memory` still stores
everything; only what `load_memory_variables` returns is windowed. With `k=1` the final question asks about a
fact that has already left the window.

In [ ]:
# ============================================================================
# PART 3.1: THE WINDOW DROPS EARLY TURNS
# ============================================================================
from langchain_classic.memory import ConversationBufferWindowMemory

window_memory = ConversationBufferWindowMemory(return_messages=True, k=1)  # last 1 exchange = 2 messages

window_chain = (
    RunnablePassthrough.assign(history=RunnableLambda(lambda _: window_memory.load_memory_variables({})["history"]))
    | prompt
    | llm
)

ask(window_chain, window_memory, "My favourite colour is teal. Explain AI in one sentence.")
ask(window_chain, window_memory, "Now do the same for deep learning.")
ask(window_chain, window_memory, "What is my favourite colour?")

print("History the next call would receive (the window has moved on):")
for m in window_memory.load_memory_variables({})["history"]:
    print(f"  {m.type}: {m.content[:70]}")
print(f"\nStored in chat_memory: {len(window_memory.chat_memory.messages)} messages · returned to the prompt: {len(window_memory.load_memory_variables({})['history'])}")


The favourite colour was only mentioned in the first exchange. With `k=1` the last call saw only the deep
learning exchange, so the model cannot know it — compare with Part 2, where the full buffer was sent.

**Choosing k**: 1–2 exchanges for quick Q&A, 3–5 for standard conversations and support, 6–10 for long
technical discussions. Whatever `k` is, early facts are silently lost once they leave the window.

### 3.2 📝 Summary memory
`ConversationSummaryMemory` replaces raw messages with a running natural-language summary written by the LLM.
It is lossy compression: details are traded for a prompt size that stays roughly constant. `save_context`
triggers the update (one extra LLM call per turn); the prompt's placeholder can have any name — here it is
`history_summary` to make the difference visible.

In [ ]:
# ============================================================================
# PART 3.2: A RUNNING SUMMARY INSTEAD OF RAW MESSAGES
# ============================================================================
from langchain_classic.memory import ConversationSummaryMemory

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", SYS_PROMPT),
    MessagesPlaceholder(variable_name="history_summary"),
    ("human", "{query}"),
])

summary_memory = ConversationSummaryMemory(return_messages=True, llm=llm)

summary_chain = (
    RunnablePassthrough.assign(
        history_summary=RunnableLambda(lambda _: summary_memory.load_memory_variables({})["history"])
    )
    | summary_prompt
    | llm
)

ask(summary_chain, summary_memory, "Explain AI in 2 bullet points.")
ask(summary_chain, summary_memory, "Now do the same for deep learning.")
print("What the next call would receive:")
print(summary_memory.load_memory_variables({})["history"])

The next prompt receives one summary message rather than four raw messages: the name and topics survive, the
exact wording does not.

### 3.3 ⚖️ Summary-buffer memory
A hybrid: recent messages stay verbatim until they exceed `max_token_limit`; then the oldest ones are folded into
a summary (`moving_summary_buffer`). You get exact recent context plus a compressed past.

In [ ]:
# ============================================================================
# PART 3.3: RECENT TURNS VERBATIM, OLDER TURNS SUMMARISED
# ============================================================================
from langchain_classic.memory import ConversationSummaryBufferMemory

review = (
    "I ordered Pizza Salami for 9.99$ and it was awesome! The pizza was delivered on time and was still hot. "
    "The crust was thin and crispy, and the toppings were fresh and flavorful. The price was reasonable and "
    "I would recommend this pizza place to others."
)

summary_buffer = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)
summary_buffer.save_context({"input": "Hi! Could you analyze a review for me?"}, {"output": "Sure, please share the review."})
summary_buffer.save_context({"input": review}, {"output": "It is a very positive review: on-time delivery, hot and fresh pizza, fair price."})

print(f"Summary of older turns:\n  {summary_buffer.moving_summary_buffer or '(none yet)'}")
print(f"\nMessages still kept verbatim: {len(summary_buffer.chat_memory.messages)}")
for m in summary_buffer.chat_memory.messages:
    print(f"  {m.type}: {m.content[:70]}")

With a 100-token limit the long review pushes the conversation over budget, so the oldest messages move into
the summary. Raise `max_token_limit` to 1000 and no summary is created at all.

| Memory class | Prompt receives | Keeps early facts | Extra LLM calls |
|---|---|---|---|
| `ConversationBufferMemory` | every message | ✅ | none |
| `ConversationBufferWindowMemory` | last `k` exchanges | ❌ | none |
| `ConversationSummaryMemory` | one summary | ✅ (compressed) | one per turn |
| `ConversationSummaryBufferMemory` | summary + recent turns | ✅ (compressed) | when over `max_token_limit` |

---
## 🔍 Part 4: Vector-Store Retriever Memory

`VectorStoreRetrieverMemory` stores each exchange as a document in a vector store and, on every call, retrieves
the top-*k* exchanges most **similar** to the new question — regardless of when they happened.

### Key Concepts:
- **Similarity, not recency**: the order of the conversation is ignored
- **`search_kwargs={"k": 2}`**: how many past exchanges to retrieve
- The memory returns one string, so it is wrapped in a list for the `MessagesPlaceholder`

In [ ]:
# ============================================================================
# PART 4A: AN EMPTY VECTOR STORE FOR PAST EXCHANGES
# ============================================================================
from aied_helpers import get_embeddings
from langchain_chroma import Chroma
from langchain_classic.memory import VectorStoreRetrieverMemory

embeddings = get_embeddings(provider="openai", model="text-embedding-3-small")
history_db = Chroma(collection_name="history_db", embedding_function=embeddings)  # in-memory collection
history_db.reset_collection()  # re-running this cell starts from an empty collection

retriever = history_db.as_retriever(search_type="similarity", search_kwargs={"k": 2})
vector_memory = VectorStoreRetrieverMemory(retriever=retriever, memory_key="history", input_key="query")

vector_chain = (
    RunnablePassthrough.assign(history=RunnableLambda(lambda x: [vector_memory.load_memory_variables(x)["history"]]))
    | prompt
    | llm
)
print("✅ Vector-store memory ready (empty)")

In [ ]:
# ============================================================================
# PART 4B: TWO TOPICS, THEN A QUESTION ABOUT THE FIRST ONE
# ============================================================================
ask(vector_chain, vector_memory, "Tell me about AI in 2 lines.")
ask(vector_chain, vector_memory, "What about deep learning?")
ask(vector_chain, vector_memory, "Tell me about the fastest animal in the world in 2 lines.")
ask(vector_chain, vector_memory, "What about the cheetah?")

# The most recent turns are about animals, but retrieval is by similarity:
print("Retrieved for 'What about machine learning?':")
print(vector_memory.load_memory_variables({"query": "What about machine learning?"})["history"])

Although the last two exchanges were about animals, the retrieved history for a machine-learning question
should be the AI and deep-learning exchanges — check the printout. That is the defining behaviour of
retriever memory: relevance over recency.

---
## 🏛️ Part 5: Legacy Conversation Chains

Before LCEL, `ConversationChain` and `LLMChain(memory=...)` did the load/save bookkeeping for you. They still
work through `langchain_classic` and are common in older code, but they are deprecated: memory is hidden inside
the chain, only one memory object is supported, and they do not compose with other runnables.

In [ ]:
# ============================================================================
# PART 5A: ConversationChain SAVES MEMORY AUTOMATICALLY
# ============================================================================
from langchain_classic.chains import ConversationChain

conversation = ConversationChain(llm=llm, memory=ConversationBufferMemory())

print(conversation.invoke({"input": "What is the capital of France?"})["response"])
print(conversation.invoke({"input": "What's the best food there?"})["response"])
print(f"\nMessages saved without calling save_context: {len(conversation.memory.chat_memory.messages)}")

In [ ]:
# ============================================================================
# PART 5B: LLMChain WITH A MESSAGES PLACEHOLDER
# ============================================================================
from langchain_classic.chains import LLMChain

chat_memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
llm_chain = LLMChain(
    llm=llm,
    prompt=ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="chat_history"),  # must match memory_key
        ("human", "{question}"),
    ]),
    memory=chat_memory,
)

llm_chain.invoke({"question": "My name is Sourav."})
print(llm_chain.invoke({"question": "What is my name?"})["text"])

Both chains remembered without any `save_context` call, but that convenience does not carry over to LCEL. Its LCEL
successor, `RunnableWithMessageHistory` (next notebook), wraps any runnable and does the load/save for you — and
is itself deprecated now in favour of a LangGraph checkpointer (module 02).

---
## 📝 Summary

In this notebook, we learned:

### 1. The classic memory API
- `ConversationBufferMemory` keeps everything; `return_messages=True` switches the output from one string to message objects

### 2. The load → invoke → save loop
- `RunnablePassthrough.assign` injects memory into an LCEL chain; `save_context` must be called by you

### 3. Bounding strategies
- Window memory returns the last `k` exchanges and forgets older facts
- Summary memory compresses the conversation with one extra LLM call per turn
- Summary-buffer memory keeps recent turns verbatim and summarises only the overflow

### 4. Vector-store memory
- Past exchanges are retrieved by similarity to the new question, not by recency

### 5. Legacy conversation chains
- `ConversationChain`/`LLMChain` saved memory automatically but are deprecated and do not compose

### Next Steps
- **`03_runnable_with_message_history.ipynb`**: automate history loading and saving for any runnable